### Path-initialized version of **missing_data_function_better**
Uses `os.getcwd()` and project folders like the model notebook.

In [97]:
# functions
import os
import sys
import shutil
import time
import pandas as pd
import warnings
import numpy as np
import math
import glob

def strToFloat(data):
    return float(str(data).replace(",", "."))

folders = ["indat", "DATAQUAL_lastAOI_missing_data_ratio"] 

current_directory = os.getcwd()

indat_path = os.path.join(current_directory, "indat")
DATAQUAL_lastAOI_missing_data_ratio_path = os.path.join(
    current_directory, "DATAQUAL_lastAOI_missing_data_ratio"
)

os.makedirs(DATAQUAL_lastAOI_missing_data_ratio_path, exist_ok=True) 

print(current_directory)
print(folders)


C:\Users\zita1\Desktop\Comp.Psych.Conf\Adatok\S1_AQ+adatok
['indat', 'DATAQUAL_lastAOI_missing_data_ratio']


In [99]:
#design
vars_dict={"randoms": 5, "epochN": 5}

In [101]:
# function to: 
##1. filter the "before_stimulus" of trial_phase variable, 
##2. filter the last before_stimulus, 
##3. get the valid and invalid samples per epoch, 
##4. compute missing data ratio (invalid/all trial) per epoch
##5. give a feedback of how many trials with missing last AOI are in each epoch/person

def computeMissingLastAOI(input_file, vars_dict):                        
    blockprepN = vars_dict["randoms"]
    epochN = vars_dict["epochN"]

    ## 1) Load + filter
    df = pd.read_csv(input_file, sep='\t', usecols=[
        'block', 'trial', 'epoch', 'trial_phase', 'left_gaze_validity', 'right_gaze_validity'
    ])
    df = df[(df['trial_phase'] == 'before_stimulus') & (df['block'] != 0)]

    ## 2) Last row per (trial)
    filename = os.path.basename(input_file)
    df_last = df.groupby(['block', 'trial'], as_index=False).last()

    ## 3) Count valid/invalid per epoch
    epoch_all = {}
    epoch_missing = {}
    for _, row in df_last.iterrows():
        trial = int(row["trial"])
        if trial <= blockprepN:
            continue  # skip preparatory trials
        epoch = int(row["epoch"])
        epoch_all[epoch] = epoch_all.get(epoch, 0) + 1
        if (not bool(row["left_gaze_validity"])) and (not bool(row["right_gaze_validity"])):
            epoch_missing[epoch] = epoch_missing.get(epoch, 0) + 1

    ## 4) Ratios (+ feedback)
    print(f"Missing trials for: {filename}")
    result = []
    for e in range(1, epochN + 1):
        if e in epoch_all:
            total = epoch_all[e]
            miss = epoch_missing.get(e, 0)
            pct = round((miss / total) * 100, 2) if total > 0 else None
            result.append(pct)
            print(f"  Epoch {e}: {miss} missing out of {total} → {pct}%")
        else:
            result.append(None)
            print(f"  Epoch {e}: no trials → None")

    return result


In [103]:
#Put the file into folder

files = glob.glob(os.path.join(indat_path, "*.txt"))

subject_epochs = []
missing_data_ratios = []

for path in files:
    fname = os.path.basename(path)
    try:
        subject = fname.split('_')[1]
    except IndexError:
        continue  # skip files not matching the expected pattern

    ratios = computeMissingLastAOI(path, vars_dict)

    for i, ratio in enumerate(ratios, start=1):
        subject_epochs.append(f"{subject}_{i}")
        missing_data_ratios.append(ratio)

df_out = pd.DataFrame({
    "subject_epoch": subject_epochs,
    "missing_ratio_percent": missing_data_ratios
})

out_csv = os.path.join(DATAQUAL_lastAOI_missing_data_ratio_path, "missinglastAOI_ratio.csv")
df_out.to_csv(out_csv, index=False)
print(f"Results saved to: {out_csv}")

Missing trials for: subject_100__log.txt
  Epoch 1: 0 missing out of 400 → 0.0%
  Epoch 2: 4 missing out of 400 → 1.0%
  Epoch 3: 8 missing out of 400 → 2.0%
  Epoch 4: 2 missing out of 400 → 0.5%
  Epoch 5: 13 missing out of 400 → 3.25%
Missing trials for: subject_101__log.txt
  Epoch 1: 1 missing out of 400 → 0.25%
  Epoch 2: 0 missing out of 400 → 0.0%
  Epoch 3: 0 missing out of 400 → 0.0%
  Epoch 4: 1 missing out of 400 → 0.25%
  Epoch 5: 0 missing out of 400 → 0.0%
Missing trials for: subject_102__log.txt
  Epoch 1: 1 missing out of 400 → 0.25%
  Epoch 2: 5 missing out of 400 → 1.25%
  Epoch 3: 2 missing out of 400 → 0.5%
  Epoch 4: 0 missing out of 400 → 0.0%
  Epoch 5: 3 missing out of 400 → 0.75%
Missing trials for: subject_104__log.txt
  Epoch 1: 0 missing out of 400 → 0.0%
  Epoch 2: 0 missing out of 400 → 0.0%
  Epoch 3: 0 missing out of 400 → 0.0%
  Epoch 4: 1 missing out of 400 → 0.25%
  Epoch 5: 1 missing out of 400 → 0.25%
Missing trials for: subject_106__log.txt
  Epoc